# CubiCasa 微調（路線圖 C）— RoomPilot-Agent

免費 T4 即可。前置（一次性）：
1. 本機跑 `python make_annotation_drafts.py`（草稿）→ **人工修正** own_dataset/ 的 model.svg → `python pack_finetune_data.py`
2. 把 `finetune_data.zip` 與基底權重 `model_best_val_loss_var.pkl` 上傳 Google Drive 根目錄
3. Runtime → Change runtime type → **T4 GPU**

訓練完 checkpoint 自動存回 Drive；帶回本機的驗收流程見最後一格。

In [ ]:
# 1) GPU 檢查
import torch
print(torch.__version__, '| CUDA:', torch.cuda.is_available(),
      '|', torch.cuda.get_device_name(0) if torch.cuda.is_available() else '無 GPU——請切 T4')

In [ ]:
# 2) 取程式庫與依賴（Colab 內建 torch/cv2/numpy）
!git clone -q https://github.com/CubiCasa/CubiCasa5k.git
!git clone -q https://github.com/Tom-Yang-Ben/RoomPilot-Agent.git 2>/dev/null || echo 'repo 私有：改用下方 wget 拿補丁腳本，或手動上傳 apply_cubicasa_patches.py'
!pip -q install lmdb svgpathtools scikit-image tensorboardX
# numpy 2.x 相容補丁（svg_utils np.matrix → np.array；含圖示樣本必須）
!python RoomPilot-Agent/apply_cubicasa_patches.py --dir CubiCasa5k

In [ ]:
# 3) Drive 掛載＋資料解壓＋基底權重
from google.colab import drive
drive.mount('/content/drive')
!unzip -q /content/drive/MyDrive/finetune_data.zip -d /content/
!cp /content/drive/MyDrive/model_best_val_loss_var.pkl /content/
!wc -l /content/finetune_data/train.txt /content/finetune_data/val.txt

In [ ]:
# 4) train.py 改吃 txt 格式（現場解析 SVG，不用 lmdb）
!sed -i "s/format='lmdb'/format='txt'/g" CubiCasa5k/train.py
!grep -n "format='txt'" CubiCasa5k/train.py

In [ ]:
# 5) 微調參數（起步值；own×3 已在 train.txt 過採樣）
L_RATE     = 1e-4   # 基底已收斂，微調用小 lr
N_EPOCH    = 20
BATCH_SIZE = 8
IMAGE_SIZE = 256    # 訓練裁切尺寸（官方預設）

In [ ]:
# 6) 微調（T4 約 1~2 小時；中斷可重跑，weights 換成最新 checkpoint 續訓）
%cd /content/CubiCasa5k
!python train.py --weights /content/model_best_val_loss_var.pkl \
    --data-path /content/finetune_data/ \
    --l-rate {L_RATE} --n-epoch {N_EPOCH} --batch-size {BATCH_SIZE} \
    --image-size {IMAGE_SIZE} --new-hyperparams
%cd /content

In [ ]:
# 7) checkpoint 存回 Drive（train.py 存 runs_cubi/ 下最新一輪）
import glob, shutil, os
ck = sorted(glob.glob('/content/CubiCasa5k/runs_cubi/**/*.pkl', recursive=True),
            key=os.path.getmtime)
assert ck, '找不到 checkpoint——檢查第 6 格是否完成'
dst = '/content/drive/MyDrive/model_finetuned_v1.pkl'
shutil.copy(ck[-1], dst)
print('已存回：', dst, '（來源：', ck[-1], '）')

In [ ]:
# 8) sanity 推論：微調後模型對一張 own 圖出房間 argmax 疊圖（目測）
import sys, torch, cv2, numpy as np
import torch.nn.functional as F
sys.path.insert(0, '/content/CubiCasa5k')
from floortrans.models import get_model

model = get_model('hg_furukawa_original', 51)
model.conv4_ = torch.nn.Conv2d(256, 44, 4, 4)   # 官方 21+12+11 頭
ckpt = torch.load('/content/drive/MyDrive/model_finetuned_v1.pkl',
                  map_location='cuda', weights_only=True)
model.load_state_dict(ckpt['model_state'])
model.eval().cuda()

img = cv2.imread('/content/finetune_data/own/floor01/F1_scaled.png')
h, w = img.shape[:2]
x = torch.tensor(cv2.cvtColor(img, cv2.COLOR_BGR2RGB), dtype=torch.float32)
x = (x.permute(2, 0, 1) / 255.0 * 2 - 1).unsqueeze(0).cuda()
with torch.no_grad():
    pred = model(F.interpolate(x, size=(512, 512), mode='bilinear'))
rooms = F.interpolate(pred[:, 21:33], size=(h, w), mode='bilinear')[0].argmax(0).cpu().numpy()
colors = np.random.RandomState(0).randint(60, 255, (12, 3), np.uint8)
vis = cv2.addWeighted(colors[rooms], 0.5, img, 0.5, 0)
from google.colab.patches import cv2_imshow
cv2_imshow(cv2.resize(vis, None, fx=0.4, fy=0.4))

## 帶回本機的驗收流程（路線 A 量尺）

1. 下載 `model_finetuned_v1.pkl` 到專案根目錄
2. 備份舊語意快取後重算評分集：`rm cubicasa_room/*_mask.npz`（或另開快取目錄）
   → `python eval_rooms_cc.py --gt-seg` 會以新權重自動重推論
   （`floorplan2room.CC_WEIGHTS` 改指新檔，或直接覆蓋檔名）
3. 對比 `eval_rooms/report_gtseg.json` 與基線（v2.7 記錄）：
   具名房型 recall 0.82~0.99 不得倒退、space precision 0.737 應改善
4. 43 題回歸：`python floorplan2room.py` → `git diff json/` 逐房檢視

**資料衛生提醒**：own 43 張只在訓練與訓練監控出現；正式驗收數字
永遠來自 val/test 評分集（train 微調樣本不重疊）。